In [1]:
import os
import json

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

from gitsource import GithubRepositoryDataReader
from evaluation_utils import llm_structured

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
# Same reader we used in homework 2 - commit 8c1834d pins the data to 72 pages
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"{len(documents)} pages loaded")

72 pages loaded


### Q1 — Generating questions

In [3]:
class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
q1_filenames = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}

q1_pages = [doc for doc in documents if doc["filename"] in q1_filenames]
print(f"{len(q1_pages)} pages selected")

3 pages selected


In [5]:
usages = []

for doc in q1_pages:
    user_prompt = json.dumps(doc)

    result, usage = llm_structured(
        client,
        data_gen_instructions,
        user_prompt,
        Questions,
    )

    usages.append(usage)
    print(doc["filename"])
    for q in result.questions:
        print("  -", q)
    print("  input_tokens:", usage.input_tokens)
    print()

01-agentic-rag/lessons/01-intro.md
  - What does RAG do to help an LLM answer questions using your own documents or FAQ data?
  - Why does the lesson say LLMs can be unreliable for fresh facts or private data?
  - What are the main steps they plan to build in the first part of the module?
  - What kind of example app are they building in this module, and what does it answer from?
  - How is the second part of the module different from the first one?
  input_tokens: 1020



01-agentic-rag/lessons/02-environment.md
  - What do I need installed before starting this module besides Python and Jupyter?
  - How do I set up a fresh project from scratch for this course, and which packages should I add?
  - What’s the recommended way to keep my API key safe so I don’t commit it by mistake?
  - How do I check that the OpenAI client is working inside a notebook?
  - If I want to use Groq or another OpenAI-style provider, what should I change in the client setup?
  input_tokens: 1286



01-agentic-rag/lessons/03-rag.md
  - Why doesn’t a normal LLM give a good answer to course-specific questions like whether I can join after the course has started?
  - What does RAG mean, and why is search part of it?
  - How do you build a prompt that uses both the student’s question and the FAQ text?
  - What should the bot reply when it can’t find the answer in the retrieved context?
  - Why does the lesson say that retrieval quality is so important in a RAG system?
  input_tokens: 1753



In [6]:
input_tokens = [usage.input_tokens for usage in usages]
avg_input_tokens = sum(input_tokens) / len(input_tokens)

print("Input tokens per call:", input_tokens)
print("Average input tokens:", avg_input_tokens)

Input tokens per call: [1020, 1286, 1753]
Average input tokens: 1353.0


## The full ground truth

In [7]:
import pandas as pd

gt_df = pd.read_csv("ground-truth.csv")
ground_truth = gt_df.to_dict(orient="records")
print(f"{len(ground_truth)} questions loaded")

360 questions loaded


## Searching the chunks

In [8]:
from gitsource import chunk_documents

# Same chunking as homework 2: 2000-char chunks, 50% overlap
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"{len(chunks)} chunks from {len(documents)} documents")

295 chunks from 72 documents


In [9]:
from minsearch import Index

tindex = Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)


def text_search(query, num_results=5):
    return tindex.search(query, num_results=num_results)


print("Text index ready")

Text index ready


In [10]:
import sys
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

project_root = Path("..").resolve()
sys.path.append(str(project_root))

from embedder import Embedder
from minsearch import VectorSearch

model_path = project_root / "models" / "Xenova" / "all-MiniLM-L6-v2"
embedder = Embedder(path=str(model_path))

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch_texts = [chunk["content"] for chunk in chunks[i : i + batch_size]]
    batch_vectors = embedder.encode_batch(batch_texts)
    vectors.extend(batch_vectors)

X = np.array(vectors)

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)


def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vindex.search(query_vector, num_results=num_results)


print(f"Vector index ready: {X.shape}")

  0%|          | 0/6 [00:00<?, ?it/s]

Vector index ready: (295, 384)


### Q2 — First result with text search

In [11]:
q = ground_truth[0]["question"]
print(q)

results = text_search(q)
print()
print("Top result filename:", results[0]["filename"])

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?

Top result filename: 01-agentic-rag/lessons/03-rag.md


### Q3 — First result with vector search

In [12]:
results = vector_search(q)
print()
print("Top result filename:", results[0]["filename"])


Top result filename: 01-agentic-rag/lessons/01-intro.md


## Evaluation metrics

In [13]:
from tqdm.auto import tqdm


def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total


def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)


def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)


def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

### Q4 — Evaluating text search

In [14]:
text_metrics = evaluate(ground_truth, text_search)
text_metrics

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

### Q5 — Evaluating vector search

In [15]:
vector_metrics = evaluate(ground_truth, vector_search)
vector_metrics

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

### Q6 — Tuning hybrid search

In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [17]:
for k in [1, 50, 100, 200]:
    metrics = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))
    print(k, metrics)

  0%|          | 0/360 [00:00<?, ?it/s]

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
